In [2]:
!pip install polars

# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')
!ls -lh "/content/drive/MyDrive/AmazonDataset/"
!cp "/content/drive/MyDrive/AmazonDataset/train_text_for_sentiment.parquet" "/content/"

Mounted at /content/drive
total 12G
-rw------- 1 root root  180 Apr  6 00:54 'báo cáo tiến độ lần 1.gdoc'
-rw------- 1 root root  180 Apr 13 00:40 'báo cáo tiến độ lần 2.gdoc'
-rw------- 1 root root 6.7G Jan 16  2025  Clothing_Shoes_and_Jewelry.jsonl.gz
-rw------- 1 root root 3.8G Jan 16  2025  meta_Clothing_Shoes_and_Jewelry.jsonl.gz
-rw------- 1 root root 1.2G Apr 29 23:30  train_text_for_sentiment.parquet


In [3]:
!ls -lh /content/

total 1.2G
drwx------ 5 root root 4.0K Apr 30 22:30 drive
drwxr-xr-x 1 root root 4.0K Apr 16 13:33 sample_data
-rw------- 1 root root 1.2G Apr 30 22:30 train_text_for_sentiment.parquet


In [4]:
import polars as pl
import os

In [5]:
# ── Paths (Tùy chỉnh theo thư mục Drive của bạn) ─────────────
BASE_DIR    = "/content"
INPUT_PATH  = f"{BASE_DIR}/train_text_for_sentiment.parquet"
OUTPUT_PATH = f"{BASE_DIR}/review_lexical.parquet"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(BASE_DIR, exist_ok=True)

In [6]:
# 1. Đọc dữ liệu từ file Parquet
df_train = pl.read_parquet(INPUT_PATH)
# 2. Hiển thị Schema để kiểm tra cấu trúc dữ liệu
print(df_train.schema)
# 4. Kiểm tra số lượng bản ghi
total_count = df_train.height
print(f"Tổng số lượng sản phẩm trong train: {total_count:,}")

# 5. Hiển thị 5 dòng đầu tiên để xem thử dữ liệu thực tế
df_train.head(5)

Schema({'mapped_user_id': Int64, 'mapped_item_id': Int64, 'rating': Float64, 'text': String})
Tổng số lượng sản phẩm trong train: 16,509,306


mapped_user_id,mapped_item_id,rating,text
i64,i64,f64,str
1358248,513151,2.0,"""Just ok Don't be fooled by the…"
1358248,325095,5.0,"""Great watch for nurses Love th…"
1358248,596094,5.0,"""FINALLY, my hat soulmate I abs…"
1358248,187347,2.0,"""Disappointing. Though these lo…"
1358248,381235,3.0,"""They're okay. Got these for my…"


In [7]:
# ════════════════════════════════════════════════════════════
# 1. KEYWORD DICTIONARY (Đã vá lỗi Partial Match \b)
# ════════════════════════════════════════════════════════════

# Negation words cần bắt
_NEG = r"(?:not|isn't|isnt|barely|hardly|never|no|don't|dont|doesn't|doesnt)"

# (column_name, keyword_regex với Word Boundary \b)
KEYWORDS: list[tuple[str, str]] = [
    # Chất liệu
    ("cotton",      r"\bcotton\b"),
    ("polyester",   r"\bpolyester\b"),
    ("leather",     r"\bleather\b"),
    ("denim",       r"\bdenim\b"),
    ("linen",       r"\blinen\b"),
    ("wool",        r"\bwool\b"),
    # Tính năng
    ("waterproof",  r"\bwaterproof\b"),
    ("breathable",  r"\bbreathable\b"),
    ("stretchy",    r"\bstretch(?:y|able)?\b"),
    # Dáng / form
    ("slim_fit",    r"\bslim[\s_-]?fit\b"),
    ("oversized",   r"\boversized\b"),
    ("regular_fit", r"\bregular[\s_-]?fit\b"),
    # Chất lượng cảm nhận
    ("soft",        r"\bsoft(?:ness)?\b"),
    ("durable",     r"\bdurab(?:le|ility)\b"),
    ("lightweight", r"\blight[\s_-]?weight\b"),
]

In [8]:
# ════════════════════════════════════════════════════════════
# 2. LOAD & CLEAN
# ════════════════════════════════════════════════════════════

print("Đang đọc dữ liệu...")
df = pl.read_parquet(INPUT_PATH)

df = df.filter(
    pl.col("text").is_not_null() &
    (pl.col("text").str.strip_chars().str.len_chars() > 0)
)

print(f"Rows loaded: {len(df):,}")

# Lowercase một lần duy nhất để tối ưu tốc độ Regex
text_lower = pl.col("text").str.to_lowercase()

Đang đọc dữ liệu...
Rows loaded: 16,509,306


In [9]:
# ════════════════════════════════════════════════════════════
# 3. STRUCTURAL FEATURES (Review Length)
# ════════════════════════════════════════════════════════════
print("Đang trích xuất độ dài văn bản...")

df = df.with_columns([
    # Đếm số từ: Số lượng khoảng trắng + 1 (Tiết kiệm RAM tuyệt đối)
    (pl.col("text").str.count_matches(r"\s+") + 1)
      .cast(pl.UInt16)
      .alias("word_count"),

    # Đếm số ký tự (Hữu ích cho XGBoost tính độ đặc của văn bản)
    pl.col("text")
      .str.len_chars()
      .cast(pl.UInt32)
      .alias("char_count")
])

Đang trích xuất độ dài văn bản...


In [10]:
# ════════════════════════════════════════════════════════════
# 4. KEYWORD FLAGS + NEGATION WINDOW (Đã vá lỗi Regex \S+)
# ════════════════════════════════════════════════════════════
print("Đang quét cờ từ khóa và phủ định...")

keyword_exprs = []

for col_name, kw_pattern in KEYWORDS:
    # ── has_[kw]: keyword xuất hiện ở bất kỳ đâu ──
    pos_expr = (
        text_lower
        .str.contains(kw_pattern)
        .cast(pl.Int8)
        .alias(f"has_{col_name}")
    )

    # ── has_not_[kw]: negation trong vòng 3 từ trước keyword ──
    # Dùng \S+ để không bị đứt gãy bởi dấu câu (phẩy, chấm)
    neg_pattern = rf"{_NEG}\s+(?:\S+\s+){{0,2}}{kw_pattern}"
    neg_expr = (
        text_lower
        .str.contains(neg_pattern)
        .cast(pl.Int8)
        .alias(f"has_not_{col_name}")
    )

    keyword_exprs.extend([pos_expr, neg_expr])

df = df.with_columns(keyword_exprs)

Đang quét cờ từ khóa và phủ định...


In [11]:
# ════════════════════════════════════════════════════════════
# 5. SELECT OUTPUT COLUMNS
# ════════════════════════════════════════════════════════════

base_cols = ["mapped_user_id", "mapped_item_id", "word_count", "char_count"]

flag_cols = []
for col_name, _ in KEYWORDS:
    flag_cols.append(f"has_{col_name}")
    flag_cols.append(f"has_not_{col_name}")

result_df = df.select(base_cols + flag_cols)

In [12]:
# ════════════════════════════════════════════════════════════
# 6. SANITY CHECK
# ════════════════════════════════════════════════════════════

print(f"\nShape output: {result_df.shape}")

# Kiểm tra tất cả flag cols đúng là Int8
for col in flag_cols:
    assert result_df[col].dtype == pl.Int8, f"FAIL: {col} không phải Int8"

# Kiểm tra flag chỉ có 0 hoặc 1
for col in flag_cols:
    assert result_df[col].is_in([0, 1]).all(), f"FAIL: {col} có giá trị ngoài {{0,1}}"

# Kiểm tra logic phủ định
for col_name, _ in KEYWORDS:
    pos_col = f"has_{col_name}"
    neg_col = f"has_not_{col_name}"
    invalid = (
        result_df
        .filter((pl.col(neg_col) == 1) & (pl.col(pos_col) == 0))
        .height
    )
    if invalid > 0:
        pct = invalid / len(result_df) * 100
        print(f"  WARN: {neg_col}=1 nhưng {pos_col}=0 → {invalid:,} rows ({pct:.2f}%)")

result_df.write_parquet(OUTPUT_PATH)
print(f"\n✅ Output lưu thành công tại: {OUTPUT_PATH}")


Shape output: (16509306, 34)

✅ Output lưu thành công tại: /content/review_lexical.parquet


In [13]:
# ── 1. Cấu hình đường dẫn ─────────────────────────────────────
# Thay đổi đường dẫn này trỏ tới file bạn vừa tạo ra
FILE_PATH = f"{BASE_DIR}/review_lexical.parquet"

# Danh sách từ khóa để test (giống ở STAGE 2)
KEYWORDS = [
    "cotton", "polyester", "leather", "denim", "linen", "wool",
    "waterproof", "breathable", "stretchy",
    "slim_fit", "oversized", "regular_fit",
    "soft", "durable", "lightweight"
]

# ── 2. Đọc và Kiểm tra cơ bản ─────────────────────────────────
print(f"Đang đọc file từ: {FILE_PATH}")
df = pl.read_parquet(FILE_PATH)

print("\n" + "="*50)
print(" 📊 BÁO CÁO KIỂM THỬ DỮ LIỆU (LEXICAL FEATURES) ")
print("="*50)

# 2.1. Kích thước và Bộ nhớ
rows, cols = df.shape
memory_mb = df.estimated_size("mb")
print(f"🔹 Số dòng (Rows) : {rows:,}")
print(f"🔹 Số cột (Cols)  : {cols}")
print(f"🔹 Dung lượng RAM : {memory_mb:.2f} MB")

# ── 3. Kiểm tra Ràng buộc Dữ liệu (Constraints Check) ─────────
print("\n" + "-"*50)
print(" 🛡️ KIỂM TRA TOÀN VẸN DỮ LIỆU (SANITY CHECKS) ")
print("-"*50)

passed_all = True

# 3.1. Kiểm tra Null
null_counts = df.null_count().to_numpy().sum()
if null_counts == 0:
    print("✅ Bảng dữ liệu sạch: Không có giá trị NULL.")
else:
    print(f"❌ CẢNH BÁO: Phát hiện {null_counts} giá trị NULL!")
    passed_all = False

# 3.2. Kiểm tra kiểu dữ liệu và giới hạn giá trị của Cờ (Flags)
for kw in KEYWORDS:
    for prefix in ["has_", "has_not_"]:
        col = f"{prefix}{kw}"

        # Kiểm tra Dtype
        if df[col].dtype != pl.Int8:
            print(f"❌ LỖI: Cột {col} không phải Int8 (Hiện tại: {df[col].dtype})")
            passed_all = False

        # Kiểm tra Max/Min phải là 0 hoặc 1
        if not df[col].is_in([0, 1]).all():
            print(f"❌ LỖI: Cột {col} chứa giá trị ngoài 0 và 1")
            passed_all = False

# 3.3. Kiểm tra Độ dài review (Không được <= 0)
if df.filter(pl.col("word_count") <= 0).height > 0:
    print("❌ LỖI: Có review đếm được số từ <= 0.")
    passed_all = False
else:
    print("✅ Cột độ dài (word_count, char_count): Hợp lệ (>0).")

if passed_all:
    print("✅ TẤT CẢ KIỂM TRA ĐỀU PASS! Dữ liệu đạt chuẩn.")

# ── 4. Thống kê Phân phối (Coverage Statistics) ────────────────
print("\n" + "-"*50)
print(" 📈 PHÂN PHỐI TỪ KHÓA (KEYWORD COVERAGE) ")
print("-"*50)

print(f"{'Từ khóa':<15} | {'Có nhắc đến (has_)':<20} | {'Phủ định (has_not_)':<20}")
print("-" * 60)

for kw in KEYWORDS:
    pos_col = f"has_{kw}"
    neg_col = f"has_not_{kw}"

    # Tính tổng số review có chứa từ khóa
    pos_count = df[pos_col].sum()
    neg_count = df[neg_col].sum()

    pos_pct = (pos_count / rows) * 100
    neg_pct = (neg_count / rows) * 100

    print(f"{kw:<15} | {pos_count:>8,} ({pos_pct:>5.2f}%) | {neg_count:>8,} ({neg_pct:>5.2f}%)")

print("\n" + "="*50)
print(" 🔍 XEM TRƯỚC DỮ LIỆU (TOP 5 DÒNG)")
print("="*50)
print(df.head(5))

Đang đọc file từ: /content/review_lexical.parquet

 📊 BÁO CÁO KIỂM THỬ DỮ LIỆU (LEXICAL FEATURES) 
🔹 Số dòng (Rows) : 16,509,306
🔹 Số cột (Cols)  : 34
🔹 Dung lượng RAM : 818.71 MB

--------------------------------------------------
 🛡️ KIỂM TRA TOÀN VẸN DỮ LIỆU (SANITY CHECKS) 
--------------------------------------------------
✅ Bảng dữ liệu sạch: Không có giá trị NULL.
✅ Cột độ dài (word_count, char_count): Hợp lệ (>0).
✅ TẤT CẢ KIỂM TRA ĐỀU PASS! Dữ liệu đạt chuẩn.

--------------------------------------------------
 📈 PHÂN PHỐI TỪ KHÓA (KEYWORD COVERAGE) 
--------------------------------------------------
Từ khóa         | Có nhắc đến (has_)   | Phủ định (has_not_) 
------------------------------------------------------------
cotton          |  167,381 ( 1.01%) |   16,377 ( 0.10%)
polyester       |   40,172 ( 0.24%) |    1,968 ( 0.01%)
leather         |  202,956 ( 1.23%) |   13,941 ( 0.08%)
denim           |   30,039 ( 0.18%) |    1,843 ( 0.01%)
linen           |    8,704 ( 0.05%) 

In [14]:
# ============================================================
# TEST CELL — review_lexical.parquet
# Sanity checks + Spot-check đầy đủ
# ============================================================

PATH = "/content/review_lexical.parquet"

df = pl.read_parquet(PATH)

KEYWORDS = [
    "cotton", "polyester", "leather", "denim", "linen", "wool",
    "waterproof", "breathable", "stretchy", "slim_fit", "oversized",
    "regular_fit", "soft", "durable", "lightweight",
]

flag_cols     = [f"has_{k}"     for k in KEYWORDS]
neg_cols      = [f"has_not_{k}" for k in KEYWORDS]
all_flag_cols = flag_cols + neg_cols

PASS = "✅ PASS"
FAIL = "❌ FAIL"

results = []   # (check_name, status, detail)

# ════════════════════════════════════════════════════════════
# BLOCK 1 — SANITY CHECKS
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("BLOCK 1 — SANITY CHECKS")
print("=" * 60)

# ── 1.1 Cột bắt buộc tồn tại ────────────────────────────────
required_cols = ["mapped_user_id", "mapped_item_id",
                 "word_count", "word_count_log1p"] + all_flag_cols
missing = [c for c in required_cols if c not in df.columns]
status  = PASS if not missing else FAIL
detail  = "tất cả cột có mặt" if not missing else f"thiếu: {missing}"
results.append(("1.1 Cột bắt buộc", status, detail))
print(f"{status} 1.1 Cột bắt buộc         — {detail}")

# ── 1.2 Dtype flag cols phải là Int8 ────────────────────────
wrong_dtype = [c for c in all_flag_cols if df[c].dtype != pl.Int8]
status = PASS if not wrong_dtype else FAIL
detail = "tất cả Int8" if not wrong_dtype else f"sai dtype: {wrong_dtype}"
results.append(("1.2 Dtype Int8", status, detail))
print(f"{status} 1.2 Dtype flag cols       — {detail}")

# ── 1.3 Không có null trong flag cols ───────────────────────
null_cols = [c for c in all_flag_cols if df[c].null_count() > 0]
status = PASS if not null_cols else FAIL
detail = "không có null" if not null_cols else f"có null: {null_cols}"
results.append(("1.3 Null flag cols", status, detail))
print(f"{status} 1.3 Null flag cols        — {detail}")

# ── 1.4 Flag chỉ có giá trị 0 hoặc 1 ───────────────────────
bad_vals = [c for c in all_flag_cols if not df[c].is_in([0, 1]).all()]
status = PASS if not bad_vals else FAIL
detail = "chỉ có {0,1}" if not bad_vals else f"giá trị lạ: {bad_vals}"
results.append(("1.4 Giá trị {0,1}", status, detail))
print(f"{status} 1.4 Giá trị {{0,1}}         — {detail}")

# ── 1.5 word_count không có null, không có giá trị âm ───────
wc_null = df["word_count"].null_count()
wc_neg  = (df["word_count"] < 0).sum()
status  = PASS if wc_null == 0 and wc_neg == 0 else FAIL
detail  = f"null={wc_null}, âm={wc_neg}"
results.append(("1.5 word_count hợp lệ", status, detail))
print(f"{status} 1.5 word_count hợp lệ     — {detail}")

# ── 1.6 word_count_log1p ~ log1p(word_count_clipped) ────────
# Kiểm tra log1p(1) ≤ word_count_log1p ≤ log1p(p99)
import math
p99      = df["word_count"].quantile(0.99)
log_min  = math.log1p(1)
log_max  = math.log1p(p99)
# out_range = df.filter(
#     (pl.col("word_count_log1p") < log_min - 1e-6) |
#     (pl.col("word_count_log1p") > log_max + 1e-6)
# ).height
# status = PASS if out_range == 0 else FAIL
# detail = f"tất cả trong [{log_min:.3f}, {log_max:.3f}]" if out_range == 0 \
#          else f"{out_range} rows ngoài range"
# results.append(("1.6 word_count_log1p range", status, detail))
# print(f"{status} 1.6 word_count_log1p      — {detail}")

# ── 1.7 has_not_[kw] = 1 nhưng has_[kw] = 0 (vi phạm logic) ─
print(f"\n  Chi tiết 1.7 — has_not=1 & has=0:")
violations_17 = []
for kw in KEYWORDS:
    bad = df.filter(
        (pl.col(f"has_not_{kw}") == 1) & (pl.col(f"has_{kw}") == 0)
    )
    if bad.height > 0:
        violations_17.append((kw, bad.height))
        note_col = "note" if "note" in df.columns else None
        sample   = bad.select(
            (["note"] if note_col else []) +
            [f"has_{kw}", f"has_not_{kw}"]
        ).head(2)
        print(f"  ⚠️  {kw}: {bad.height} row(s)")
        print(f"      {sample}")

status = PASS if not violations_17 else FAIL
detail = "không vi phạm" if not violations_17 \
         else f"vi phạm: {[v[0] for v in violations_17]}"
results.append(("1.7 has_not ≤ has logic", status, detail))
print(f"\n{status} 1.7 has_not ≤ has logic   — {detail}")

# ── 1.8 Không có duplicate key hoàn toàn giống nhau ─────────
n_dup = df.height - df.unique(
    subset=["mapped_user_id", "mapped_item_id"] + all_flag_cols
).height
status = PASS if n_dup == 0 else "⚠️  WARN"
detail = f"{n_dup} rows hoàn toàn trùng nhau" if n_dup > 0 \
         else "không có duplicate hoàn toàn"
results.append(("1.8 Duplicate rows", status, detail))
print(f"{status} 1.8 Duplicate rows         — {detail}")


# ════════════════════════════════════════════════════════════
# BLOCK 2 — SPOT-CHECK THEO EDGE CASE
# Chỉ chạy nếu có cột 'note' (file test)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BLOCK 2 — SPOT-CHECK EDGE CASES")
print("=" * 60)

if "note" not in df.columns:
    print("⚠️  Không có cột 'note' — bỏ qua Block 2 (file production)")
else:
    # Định nghĩa expected values cho từng note
    # Format: (note_contains, col, expected_value, mo_ta)
    expected_checks = [
        # Normal cases
        ("normal: cotton+soft",        "has_cotton",        1, "cotton detect"),
        ("normal: cotton+soft",        "has_soft",          1, "soft detect"),
        ("normal: cotton+soft",        "has_lightweight",   1, "lightweight detect"),
        ("normal: leather+durable",    "has_leather",       1, "leather detect"),
        ("normal: leather+durable",    "has_durable",       1, "durable detect"),
        ("normal: waterproof",         "has_waterproof",    1, "waterproof detect"),
        ("normal: waterproof",         "has_breathable",    1, "breathable detect"),

        # Negation cases
        ("negation: not cotton",       "has_cotton",        1, "cotton vẫn xuất hiện"),
        ("negation: not cotton",       "has_not_cotton",    1, "not cotton detect"),
        ("negation: not waterproof",   "has_waterproof",    1, "waterproof vẫn xuất hiện"),
        ("negation: not waterproof",   "has_not_waterproof",1, "not waterproof detect"),
        ("negation: never soft",       "has_not_soft",      1, "never soft detect"),
        ("negation: barely stretchy",  "has_not_stretchy",  1, "barely stretchy detect"),
        ("negation: isn't leather",    "has_not_leather",   1, "isn't leather detect"),

        # Partial match — word boundary
        ("partial: softball",          "has_soft",          0, "softball ≠ soft (word boundary)"),
        ("partial: software",          "has_soft",          0, "software ≠ soft (word boundary)"),
        ("partial: software",          "has_cotton",        1, "cotton vẫn detect trong câu software"),

        # Uppercase
        ("uppercase: cần lowercase",   "has_cotton",        1, "COTTON uppercase detect"),
        ("uppercase negation",         "has_not_waterproof",1, "NOT WATERPROOF uppercase detect"),

        # Multi-word keyword
        ("slim_fit với space",         "has_slim_fit",      1, "slim fit space detect"),
        ("slim_fit với hyphen",        "has_slim_fit",      1, "slim-fit hyphen detect"),
        ("negation slim_fit",          "has_not_slim_fit",  1, "not slim fit detect"),

        # Negation window dài (4 từ — ngoài window 3 từ)
        ("negation window 4 từ",       "has_not_leather",   0, "window 4 từ không bắt được — đúng"),

        # Cross-sentence (không nên bắt negation)
        ("cross-sentence: not [.]",    "has_leather",       1, "leather vẫn detect"),
        ("cross-sentence: not [.]",    "has_not_leather",   0, "cross-sentence không false positive"),

        # word_count log1p
        ("long: 302 từ",               "word_count_log1p",  None, "log1p < log1p(p99) — clip hoạt động"),
    ]

    spot_pass = spot_fail = spot_warn = 0

    for (note_kw, col, expected, mo_ta) in expected_checks:
        rows = df.filter(pl.col("note").str.contains(note_kw))

        if rows.height == 0:
            print(f"  ⚠️  SKIP  [{note_kw}] — không tìm thấy row")
            spot_warn += 1
            continue

        if expected is None:
            # Kiểm tra đặc biệt: log1p clip
            p99_wc   = df["word_count"].quantile(0.99)
            max_log  = rows["word_count_log1p"].max()
            expected_max = math.log1p(p99_wc)
            ok = max_log <= expected_max + 1e-6
            sym = "✅" if ok else "❌"
            print(f"  {sym}  [{note_kw}] {mo_ta}: log1p={max_log:.4f} ≤ {expected_max:.4f}")
            if ok: spot_pass += 1
            else:  spot_fail += 1
            continue

        actual = rows[col].to_list()
        all_ok = all(v == expected for v in actual)
        sym    = "✅" if all_ok else "❌"
        detail = f"expected={expected}, got={actual}"
        print(f"  {sym}  [{note_kw[:30]:<30}] {mo_ta}: {detail}")
        if all_ok: spot_pass += 1
        else:      spot_fail += 1

    print(f"\nSpot-check: {spot_pass} pass, {spot_fail} fail, {spot_warn} skip")
    results.append((
        "2. Spot-check edge cases",
        PASS if spot_fail == 0 else FAIL,
        f"{spot_pass}/{spot_pass+spot_fail} checks passed"
    ))


# ════════════════════════════════════════════════════════════
# BLOCK 3 — TỔNG KẾT
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("TỔNG KẾT")
print("=" * 60)
total_fail = sum(1 for _, s, _ in results if "FAIL" in s)
total_warn = sum(1 for _, s, _ in results if "WARN" in s)
for name, status, detail in results:
    print(f"  {status}  {name:<35} {detail}")
print(f"\n{'🎉 Tất cả checks passed!' if total_fail == 0 else f'⛔ {total_fail} check(s) FAILED'}"
      + (f"  ({total_warn} warning)" if total_warn else ""))

# ── Thống kê coverage nhanh ──────────────────────────────────
print("\n── Keyword coverage (% rows có keyword) ──")
total = df.height
for kw in KEYWORDS:
    pos = df[f"has_{kw}"].sum()
    neg = df[f"has_not_{kw}"].sum()
    bar = "█" * int(pos / total * 20)
    print(f"  {kw:<14} has={pos/total*100:5.1f}% {bar:<20}  has_not={neg/total*100:4.1f}%")

BLOCK 1 — SANITY CHECKS
❌ FAIL 1.1 Cột bắt buộc         — thiếu: ['word_count_log1p']
✅ PASS 1.2 Dtype flag cols       — tất cả Int8
✅ PASS 1.3 Null flag cols        — không có null
✅ PASS 1.4 Giá trị {0,1}         — chỉ có {0,1}
✅ PASS 1.5 word_count hợp lệ     — null=0, âm=0

  Chi tiết 1.7 — has_not=1 & has=0:

✅ PASS 1.7 has_not ≤ has logic   — không vi phạm
⚠️  WARN 1.8 Duplicate rows         — 175799 rows hoàn toàn trùng nhau

BLOCK 2 — SPOT-CHECK EDGE CASES
⚠️  Không có cột 'note' — bỏ qua Block 2 (file production)

TỔNG KẾT
  ❌ FAIL  1.1 Cột bắt buộc                    thiếu: ['word_count_log1p']
  ✅ PASS  1.2 Dtype Int8                      tất cả Int8
  ✅ PASS  1.3 Null flag cols                  không có null
  ✅ PASS  1.4 Giá trị {0,1}                   chỉ có {0,1}
  ✅ PASS  1.5 word_count hợp lệ               null=0, âm=0
  ✅ PASS  1.7 has_not ≤ has logic             không vi phạm
  ⚠️  WARN  1.8 Duplicate rows                  175799 rows hoàn toàn trùng nhau

⛔ 1 check(